# src_v5 — Image Masking Pipeline

Three **specialist detectors**, one **shared coordinate stage**, a **CLIP verification gate**, and a **debug overlay + IoU eval harness**. The design fixes v4's central defect — masks landing in the wrong location — and adds the missing face category.

| Content | Model | Notes |
| ------- | ----- | ----- |
| **Logos** (graphical emblems) | Grounding DINO `IDEA-Research/grounding-dino-base` + CLIP gate | Correct `(H,W)` post-process → pixel-accurate boxes. CLIP drops decorative icons; reference-image similarity **keeps Databricks' own logo**. |
| **Faces** | OpenCV **YuNet** | Bundled in opencv, no extra deps; blurred (not blacked) by default. |
| **Text / wordmark logos / PII** | Databricks `ai_parse_document` (+ regex/Claude PII filter) | Runs on the Databricks deployment; padded boxes + regex fallback fix v4's leaks/over-masking. |

See `EVALUATION.md` (why v4 failed) and `PLAN.md` (the full design). Detector code lives in `pipeline/`.

> **Environment note:** logos + faces run fully locally (CPU/MPS/GPU). The text phase needs a Databricks serverless session (`ai_parse_document`); set `DO_TEXT=True` when running on-platform with a valid profile.

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import os, sys, glob, time
sys.path.insert(0, os.path.dirname(os.path.abspath('')) if '__file__' not in dir() else '.')
# Ensure the src_v5 dir (containing pipeline/) is importable:
SRC_V5 = os.path.abspath('.')
if 'src_v5' not in SRC_V5:
    SRC_V5 = os.path.join(SRC_V5, 'image_masking', 'src_v5')
sys.path.insert(0, SRC_V5)

from PIL import Image
from pipeline.run import MaskingPipeline, PipelineConfig
from pipeline.detectors import pick_device
print('device:', pick_device())

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
IMAGES_DIR = os.path.join(os.path.dirname(SRC_V5), 'images')
OUT_DIR    = os.path.join(SRC_V5, 'outputs')
os.makedirs(OUT_DIR, exist_ok=True)

# Toggle the text phase on when running on Databricks (needs ai_parse_document).
DO_TEXT = False
DATABRICKS_PROFILE = 'e2-field-eng-west'
CLAUDE_ENDPOINT = (
    'https://e2-demo-field-eng.cloud.databricks.com'
    '/serving-endpoints/databricks-claude-sonnet-4/invocations'
)

config = PipelineConfig(
    do_logos=True,
    do_faces=True,
    do_text=DO_TEXT,
    use_clip_gate=True,        # drop decorative-icon false positives; keep Databricks logos
    text_mode='pii_only',      # 'pii_only' | 'all_text'
    logo_box_threshold=0.25,
    logo_text_threshold=0.20,
    face_pad_frac=0.15,
)

# Mask style is set per-source in pipeline/masking.py: faces→blur, logos/text→black.
# Override here if desired, e.g. STYLE_OVERRIDES = {'face': 'black'}.
STYLE_OVERRIDES = None

In [ ]:
# ── Initialise pipeline (optionally with a Databricks session for text) ───────
spark = None
if DO_TEXT:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.profile(DATABRICKS_PROFILE).serverless(True).getOrCreate()
    print('Spark:', spark.version)

pipe = MaskingPipeline(
    config,
    spark=spark,
    claude_endpoint=CLAUDE_ENDPOINT if DO_TEXT else None,
)
print('Pipeline ready. Models load lazily on first use.')

In [ ]:
# ── Discover source images (skip previously-masked outputs) ───────────────────
exts = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tiff', '*.webp')
image_paths = sorted(
    p for e in exts for p in glob.glob(os.path.join(IMAGES_DIR, e))
    if '_masked' not in os.path.basename(p)
)
print(f'{len(image_paths)} source images')
for p in image_paths:
    print('  ', os.path.basename(p))

In [ ]:
# ── Run the pipeline → debug overlay + masked output per image ────────────────
from pipeline.masking import apply_masks
from pipeline.overlay import render_overlay

all_results = {}
for path in image_paths:
    t = time.time()
    image = Image.open(path).convert('RGB')
    dets = pipe.detect(image, path)
    overlay = render_overlay(image, dets)
    masked  = apply_masks(image, dets, style_overrides=STYLE_OVERRIDES)

    stem = os.path.splitext(os.path.basename(path))[0]
    overlay.save(os.path.join(OUT_DIR, f'{stem}_overlay.jpg'))
    masked.save(os.path.join(OUT_DIR,  f'{stem}_masked.jpg'), quality=95)
    all_results[os.path.basename(path)] = dets

    n_mask = sum(1 for d in dets if d.mask)
    kept   = [d.label for d in dets if not d.mask]
    by = {}
    for d in dets:
        if d.mask: by[d.source] = by.get(d.source, 0) + 1
    print(f'{os.path.basename(path):32s} {time.time()-t:5.1f}s  mask={n_mask} {by}'
          + (f'  KEPT={kept}' if kept else ''))

In [ ]:
# ── Inspect one result (overlay shows WHAT was found + WHERE; masked = output) ─
import matplotlib.pyplot as plt
stem = os.path.splitext(os.path.basename(image_paths[0]))[0]
fig, ax = plt.subplots(1, 2, figsize=(18, 9))
ax[0].imshow(Image.open(os.path.join(OUT_DIR, f'{stem}_overlay.jpg'))); ax[0].set_title('Debug overlay'); ax[0].axis('off')
ax[1].imshow(Image.open(os.path.join(OUT_DIR, f'{stem}_masked.jpg')));  ax[1].set_title('Masked output'); ax[1].axis('off')
plt.tight_layout(); plt.show()

## Evaluation harness

Quantify localization instead of eyeballing it. Per category: **precision / recall @ IoU≥0.5** and **coverage** (mean fraction of each ground-truth box covered by a mask).

Workflow (per the auto-propose decision):
1. `auto_propose_ground_truth(...)` writes the pipeline's own detections to `eval/ground_truth.json` as a **starting scaffold**.
2. **You correct it** — fix boxes, add missed items, delete false positives. (Auto-proposed GT scores ~perfectly against itself; correction is what makes the numbers meaningful.)
3. `score_set(...)` reports metrics; re-run after every threshold change.

In [ ]:
# ── (1) Auto-propose a ground-truth scaffold to correct ───────────────────────
from eval.score import auto_propose_ground_truth, score_set
GT_PATH = os.path.join(SRC_V5, 'eval', 'ground_truth.json')
# Uncomment to (re)generate the scaffold from current detections:
# auto_propose_ground_truth(pipe, image_paths, GT_PATH)
# print('Wrote scaffold ->', GT_PATH, '— now correct it by hand.')

In [ ]:
# ── (3) Score current detections against (corrected) ground truth ─────────────
import json
report = score_set(all_results, GT_PATH, iou_thr=0.5)
print(json.dumps(report['summary'], indent=2))

## MLflow tracking (model + workflow performance)

Capture each evaluation as an **MLflow run** — params (config, model ids, git sha), metrics (per-category precision/recall/coverage + workflow latency / mask-area / detection counts), and artifacts (overlays, masked images, report). One config = one run, so threshold sweeps and model swaps compare directly in the MLflow UI. On Databricks MLflow is auto-configured; locally with no MLflow it runs in dry-run mode (metrics computed, not logged).

See `pipeline/tracking.py` and the headless workflow `eval/track_masking.py` (runnable as a Databricks job).

In [ ]:
# ── Log one evaluation run ────────────────────────────────────────────────────
from pipeline.tracking import evaluate_and_log

payload = evaluate_and_log(
    pipe, image_paths, GT_PATH,
    experiment='/Shared/image-masking-v5',   # workspace path on Databricks
    run_name='baseline',
    iou_thr=0.5,
    log_images=True,
)
import json; print(json.dumps(payload['metrics'], indent=2, default=str))

In [ ]:
# ── Threshold sweep — one MLflow run per logo confidence threshold ────────────
from pipeline.tracking import sweep_and_log

def make_pipeline(overrides):
    cfg = PipelineConfig(do_logos=True, do_faces=True, do_text=DO_TEXT,
                         use_clip_gate=True, **{k: v for k, v in overrides.items()
                                                if not k.startswith('_')})
    return MaskingPipeline(cfg, spark=spark,
                           claude_endpoint=CLAUDE_ENDPOINT if DO_TEXT else None)

configs = [
    {'logo_box_threshold': 0.20, '_run_name': 'logo_thr_0.20'},
    {'logo_box_threshold': 0.25, '_run_name': 'logo_thr_0.25'},
    {'logo_box_threshold': 0.30, '_run_name': 'logo_thr_0.30'},
]
# sweep_and_log(make_pipeline, configs, image_paths, GT_PATH,
#               experiment='/Shared/image-masking-v5', iou_thr=0.5)